In [ ]:
#lstm forecast

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM

# Actual data from 2006 to 2025
years = list(range(2006, 2026))
remaining_forest_percentages = [
    19.43, 19.00, 19.37, 19.75, 19.46, 18.53, 18.67,
    18.24, 18.26, 18.45, 18.52, 18.28, 18.26,
    18.37, 18.09, 18.37, 18.57, 18.18, 18.38, 18.03
]

# Training data up to 2023
training_years = list(range(2006, 2024))
training_data = remaining_forest_percentages[:len(training_years)]
test_data = remaining_forest_percentages[len(training_years):]  # Actual values for 2024–2025

# Normalize the data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(np.array(training_data).reshape(-1, 1))

# Prepare sequences (lookback = 3)
lookback = 10
X, y = [], []
for i in range(lookback, len(scaled_data)):
    X.append(scaled_data[i - lookback:i, 0])
    y.append(scaled_data[i, 0])
X, y = np.array(X), np.array(y)
X = X.reshape((X.shape[0], X.shape[1], 1))

# Build LSTM model
model = Sequential([
    LSTM(50, return_sequences=False, input_shape=(lookback, 1)),
    Dense(1)
])
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X, y, epochs=200, batch_size=16, verbose=0)

# Forecast for 2024 and 2025
forecast_scaled = []
input_seq = scaled_data[-lookback:].reshape(1, lookback, 1)

for _ in range(2):  # Forecast 2 steps ahead
    pred = model.predict(input_seq, verbose=0)[0][0]
    forecast_scaled.append(pred)
    input_seq = np.append(input_seq[:, 1:, :], [[[pred]]], axis=1)

# Inverse transform forecasted values
forecast = scaler.inverse_transform(np.array(forecast_scaled).reshape(-1, 1)).flatten()
forecast_years = [2024, 2025]

# Plotting
plt.figure(figsize=(10, 5))

# Actual data
plt.plot(years, remaining_forest_percentages, marker='o', color='blue', label='Actual (2006–2025)')
for x, y_val in zip(years, remaining_forest_percentages):
    plt.text(x, y_val - 0.2, f'{y_val:.2f}', ha='center', va='bottom', fontsize=8, color='blue')

# Forecasted values
plt.plot(forecast_years, forecast, marker='o', linestyle='--', color='green', label='LSTM Forecast (2024–2025)')
for x, y_val in zip(forecast_years, forecast):
    plt.text(x, y_val + 0.3, f'{y_val:.2f}', ha='center', va='top', fontsize=8, color='green')

plt.axvline(x=2024, linestyle='--', color='gray', label='Forecast Start')

plt.title("LSTM Forecast of Remaining Forest Percentage in Sundarban (2006–2025)")
plt.xlabel("Year")
plt.ylabel("Forest Percentage")
plt.xticks(ticks=range(2006, 2026), rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Print forecast vs actual
print("\nLSTM Forecast vs Actual (2024–2025):")
for i, year in enumerate(forecast_years):
    predicted = forecast[i]
    actual = test_data[i]
    diff = abs(predicted - actual)
    comment = "very close!" if diff < 0.05 else "a small gap"
    print(f"{year}: Forecast = {predicted:.2f}%, Actual = {actual:.2f}%, "
          f"Difference = {diff:.2f}% ({comment})")


In [ ]:
# design lstm graph based on lstm forecast

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# -------------------------------
# Clay2 style
# -------------------------------
plt.rcParams['legend.handlelength'] = 0
plt.rcParams['legend.numpoints'] = 1
plt.rcParams['lines.linewidth'] = 2
plt.rcParams["font.family"] = "Arial"
plt.rcParams['font.size'] = 18
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Arial"
plt.rcParams["mathtext.it"] = "Arial:italic"
plt.rcParams["mathtext.bf"] = "Arial:bold"

# -------------------------------
# Data
# -------------------------------
years = list(range(2006, 2026))
remaining_forest_percentages = [
    19.43, 19.00, 19.37, 19.75, 19.46, 18.53, 18.67,
    18.24, 18.26, 18.45, 18.52, 18.28, 18.26,
    18.37, 18.09, 18.37, 18.57, 18.18, 18.38, 18.03
]

# Forecast values (manual)
forecast_years = [2024, 2025]
forecast = [18.25, 18.30]



# -------------------------------
# Plot
# -------------------------------
fig, ax = plt.subplots(figsize=(11, 6))

# Actual data
ax.plot(years, remaining_forest_percentages, '-o', color='blue',
        label='Actual (2006–2025)', markersize=5)

# Forecasted values
ax.plot(forecast_years, forecast, '--o', color='green',
        label='Forecast (2024–2025)', markersize=7)

# Annotate points
for x, y_val in zip(years, remaining_forest_percentages):
    ax.text(x, y_val - 0.09, f'{y_val:.2f}', ha='center', va='top', fontsize=9, color='blue')

for x, y_val in zip(forecast_years, forecast):
    ax.text(x, y_val + 0.3, f'{y_val:.2f}', ha='center', va='bottom', fontsize=10,
            color='green', fontweight='bold')

# Vertical line for forecast start
ax.axvline(x=2024, linestyle='--', color='gray', label='Forecast Start')

# Labels and axis
ax.set_xlabel("Year", fontsize=20)
ax.set_ylabel("Forest Percentage (%)", fontsize=20)
ax.set_xlim(2005.5, 2025.5)
ax.set_xticks(range(2006, 2026, 2))  # every 2 years
ax.set_xticklabels(range(2006, 2026, 2), rotation=45)

# Ticks styling
ax.tick_params(axis='both', which='major', direction='out', length=6, width=2,
               colors='black', labelsize=14)
ax.tick_params(axis='both', which='minor', direction='out', length=4, width=2,
               colors='black', labelsize=14)
ax.xaxis.set_minor_locator(plt.MultipleLocator(1))
ax.yaxis.set_minor_locator(plt.MultipleLocator(0.5))

# Thicken axes
for spine in ax.spines.values():
    spine.set_linewidth(2)

# Legend
ax.legend(frameon=False, fontsize=16)

plt.tight_layout()
plt.show()

# -------------------------------
# Print forecast vs actual
# -------------------------------
test_data = remaining_forest_percentages[-2:]  # Actual 2024-2025
print("\nForecast vs Actual (2024–2025):")
for i, year in enumerate(forecast_years):
    predicted = forecast[i]
    actual = test_data[i]
    diff = abs(predicted - actual)
    comment = "very close!" if diff < 0.05 else "a small gap"
    print(f"{year}: Forecast = {predicted:.2f}%, Actual = {actual:.2f}%, "
          f"Difference = {diff:.2f}% ({comment})")


In [ ]:
#Convlstm forecast

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, Flatten, Dense

# Actual data from 2006 to 2025
years = list(range(2006, 2026))
remaining_forest_percentages = [
    19.43, 19.00, 19.37, 19.75, 19.46, 18.53, 18.67,
    18.24, 18.26, 18.45, 18.52, 18.28, 18.26,
    18.37, 18.09, 18.37, 18.57, 18.18, 18.38, 18.03
]

# Training data up to 2023
training_years = list(range(2006, 2024))
training_data = remaining_forest_percentages[:len(training_years)]
test_data = remaining_forest_percentages[len(training_years):]  # Actual values for 2024–2025

# Normalize the data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(np.array(training_data).reshape(-1, 1))

# Prepare sequences (lookback = 3)
lookback = 5
X, y = [], []
for i in range(lookback, len(scaled_data)):
    X.append(scaled_data[i - lookback:i])
    y.append(scaled_data[i, 0])

X, y = np.array(X), np.array(y)

# Reshape to 5D for ConvLSTM2D: (samples, time steps, rows, cols, channels)
X = X.reshape((X.shape[0], lookback, 1, 1, 1))

# Build ConvLSTM model
model = Sequential([
    ConvLSTM2D(filters=32, kernel_size=(1, 1), activation='relu',
               input_shape=(lookback, 1, 1, 1), return_sequences=False),
    Flatten(),
    Dense(1)
])
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X, y, epochs=200, batch_size=16, verbose=0)

# Forecast for 2024 and 2025
forecast_scaled = []
last_sequence = scaled_data[-lookback:].reshape((1, lookback, 1, 1, 1))

for _ in range(2):
    pred = model.predict(last_sequence, verbose=0)[0][0]
    forecast_scaled.append(pred)
    # Update sequence with the new prediction
    new_step = np.array(pred).reshape(1, 1, 1, 1, 1)
    last_sequence = np.concatenate((last_sequence[:, 1:], new_step), axis=1)

# Inverse transform forecasted values
forecast = scaler.inverse_transform(np.array(forecast_scaled).reshape(-1, 1)).flatten()
forecast_years = [2024, 2025]

# Plotting
plt.figure(figsize=(10, 5))

# Actual data
plt.plot(years, remaining_forest_percentages, marker='o', color='blue', label='Actual (2006–2025)')
for x, y_val in zip(years, remaining_forest_percentages):
    plt.text(x, y_val - 0.2, f'{y_val:.2f}', ha='center', va='bottom', fontsize=8, color='blue')

# Forecasted values
plt.plot(forecast_years, forecast, marker='o', linestyle='--', color='purple', label='ConvLSTM Forecast (2024–2025)')
for x, y_val in zip(forecast_years, forecast):
    plt.text(x, y_val + 0.3, f'{y_val:.2f}', ha='center', va='top', fontsize=8, color='purple')

plt.axvline(x=2024, linestyle='--', color='gray', label='Forecast Start')

plt.title("ConvLSTM Forecast of Remaining Forest Percentage in Sundarban (2006–2025)")
plt.xlabel("Year")
plt.ylabel("Forest Percentage")
plt.xticks(ticks=range(2006, 2026), rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Print forecast vs actual
print("\nConvLSTM Forecast vs Actual (2024–2025):")
for i, year in enumerate(forecast_years):
    predicted = forecast[i]
    actual = test_data[i]
    diff = abs(predicted - actual)
    comment = "very close!" if diff < 0.05 else "a small gap"
    print(f"{year}: Forecast = {predicted:.2f}%, Actual = {actual:.2f}%, "
          f"Difference = {diff:.2f}% ({comment})")


In [ ]:
# design convlstm graph based on convlstm forecast

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------
# Clay2 style
# -------------------------------
plt.rcParams['legend.handlelength'] = 0
plt.rcParams['legend.numpoints'] = 1
plt.rcParams['lines.linewidth'] = 2
plt.rcParams["font.family"] = "Arial"
plt.rcParams['font.size'] = 18
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Arial"
plt.rcParams["mathtext.it"] = "Arial:italic"
plt.rcParams["mathtext.bf"] = "Arial:bold"

# -------------------------------
# Data
# -------------------------------
years = list(range(2006, 2026))
remaining_forest_percentages = [
    19.43, 19.00, 19.37, 19.75, 19.46, 18.53, 18.67,
    18.24, 18.26, 18.45, 18.52, 18.28, 18.26,
    18.37, 18.09, 18.37, 18.57, 18.18, 18.38, 18.03
]

# Forecast values (manual)
forecast_years = [2024, 2025]
forecast = [18.28, 18.32]


# -------------------------------
# Plot
# -------------------------------
fig, ax = plt.subplots(figsize=(11, 6))

# Actual data
ax.plot(years, remaining_forest_percentages, '-o', color='blue',
        label='Actual (2006–2025)', markersize=5)

# Forecasted values
ax.plot(forecast_years, forecast, '--o', color='purple',
        label='Forecast (2024–2025)', markersize=7)

# Annotate points
for x, y_val in zip(years, remaining_forest_percentages):
    ax.text(x, y_val - 0.12, f'{y_val:.2f}', ha='center', va='top', fontsize=9, color='blue')

for x, y_val in zip(forecast_years, forecast):
    ax.text(x, y_val + 0.3, f'{y_val:.2f}', ha='center', va='bottom', fontsize=10,
            color='purple', fontweight='bold')

# Vertical line for forecast start
ax.axvline(x=2024, linestyle='--', color='gray', label='Forecast Start')

# Labels and axis
ax.set_xlabel("Year", fontsize=20)
ax.set_ylabel("Forest Percentage (%)", fontsize=20)
ax.set_xlim(2005.5, 2025.5)
ax.set_xticks(range(2006, 2026, 2))  # every 2 years
ax.set_xticklabels(range(2006, 2026, 2), rotation=45)

# Ticks styling
ax.tick_params(axis='both', which='major', direction='out', length=6, width=2,
               colors='black', labelsize=14)
ax.tick_params(axis='both', which='minor', direction='out', length=4, width=2,
               colors='black', labelsize=14)
ax.xaxis.set_minor_locator(plt.MultipleLocator(1))
ax.yaxis.set_minor_locator(plt.MultipleLocator(0.5))

# Thicken axes
for spine in ax.spines.values():
    spine.set_linewidth(2)

# Legend
ax.legend(frameon=False, fontsize=16)

plt.tight_layout()
plt.show()

# -------------------------------
# Print forecast vs actual
# -------------------------------
test_data = remaining_forest_percentages[-2:]  # Actual 2024-2025
print("\nForecast vs Actual (2024–2025):")
for i, year in enumerate(forecast_years):
    predicted = forecast[i]
    actual = test_data[i]
    diff = abs(predicted - actual)
    comment = "very close!" if diff < 0.05 else "a small gap"
    print(f"{year}: Forecast = {predicted:.2f}%, Actual = {actual:.2f}%, "
          f"Difference = {diff:.2f}% ({comment})")


In [ ]:
#Comparison Plot between arima, lstm and convlstm.
#Get your data for arima, lstm and convlstm and paste it here

In [ ]:
import matplotlib.pyplot as plt

# -------------------------------
# Clay2 style
# -------------------------------
plt.rcParams['legend.handlelength'] = 0
plt.rcParams['legend.numpoints'] = 1
plt.rcParams['lines.linewidth'] = 2
plt.rcParams["font.family"] = "Arial"
plt.rcParams['font.size'] = 18
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Arial"
plt.rcParams["mathtext.it"] = "Arial:italic"
plt.rcParams["mathtext.bf"] = "Arial:bold"

# -------------------------------
# Data
# -------------------------------
years = list(range(2006, 2026))

# Actual values
actual = [
     19.43, 19.00, 19.37, 19.75, 19.46, 18.53, 18.67,
    18.24, 18.26, 18.45, 18.52, 18.28, 18.26,
    18.37, 18.09, 18.37, 18.57, 18.18, 18.38, 18.03
]



# Forecasts for 2024–2025 (manual)
lstm_forecast = [18.85, 18.86]
convlstm_forecast = [18.76, 18.78]
arima_forecast = [18.96, 19.00]

# Extend forecasts to full length (None for earlier years)
lstm_full = [None]*18 + lstm_forecast
convlstm_full = [None]*18 + convlstm_forecast
arima_full = [None]*18 + arima_forecast



# -------------------------------
# Plot
# -------------------------------
fig, ax = plt.subplots(figsize=(12, 6))

# Actual data
ax.plot(years, actual, '-o', color='blue', label='Actual', markersize=5)

# Forecasts
ax.plot(years, lstm_full, '--o', color='green', label='LSTM Forecast', markersize=6)
ax.plot(years, convlstm_full, '--o', color='purple', label='ConvLSTM Forecast', markersize=6)
ax.plot(years, arima_full, '--o', color='orange', label='ARIMA Forecast', markersize=6)

# Annotate points
for x, y_val in zip(years, actual):
    ax.text(x, y_val - 0.25, f'{y_val:.2f}', ha='center', va='top', fontsize=9, color='blue')

for x, y_val in zip(years[-2:], lstm_forecast):
    ax.text(x, y_val + 0.2, f'{y_val:.2f}', ha='center', va='bottom', fontsize=10, color='green', fontweight='bold')

for x, y_val in zip(years[-2:], convlstm_forecast):
    ax.text(x, y_val + 0.4, f'{y_val:.2f}', ha='center', va='bottom', fontsize=10, color='purple', fontweight='bold')

for x, y_val in zip(years[-2:], arima_forecast):
    ax.text(x, y_val + 0.4, f'{y_val:.2f}', ha='center', va='bottom', fontsize=10, color='orange', fontweight='bold')

# Vertical line for forecast start
ax.axvline(x=2024, linestyle='--', color='gray', label='Forecast Start')

# Labels and axis
ax.set_xlabel("Year", fontsize=20)
ax.set_ylabel("Remaining Forest Percentage (%)", fontsize=20)
ax.set_xlim(2005.5, 2025.5)
ax.set_xticks(range(2006, 2026, 2))  # every 2 years
ax.set_xticklabels(range(2006, 2026, 2), rotation=45)

# Ticks styling
ax.tick_params(axis='both', which='major', direction='out', length=6, width=2, colors='black', labelsize=14)
ax.tick_params(axis='both', which='minor', direction='out', length=4, width=2, colors='black', labelsize=14)
ax.xaxis.set_minor_locator(plt.MultipleLocator(1))
ax.yaxis.set_minor_locator(plt.MultipleLocator(0.5))

# Thicken axes
for spine in ax.spines.values():
    spine.set_linewidth(2)

# Legend
ax.legend(frameon=False, fontsize=16)

plt.tight_layout()
plt.show()
